Data obtained here: https://zenodo.org/records/14767363  
egrid details here: https://www.epa.gov/system/files/documents/2025-01/egrid2023_technical_guide.pdf  
ejscreen in action here: https://pedp-ejscreen.azurewebsites.net/  
ejscreen documentation here: https://www.epa.gov/system/files/documents/2024-07/ejscreen-tech-doc-version-2-3.pdf  
other resource i couldnt figure out: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/RLR5AX  
ejscreen tool archive: https://screening-tools.com/epa-ejscreen  


In [9]:
import pandas as pd


In [10]:
use_cols = [
    'ID', 
    'PEOPCOLOR', 
    'ACSTOTPOP', #Total population
    'LOWINCOME', 'ACSIPOVBAS', #Population for whom poverty status is determined
    'UNEMPLOYED',  'ACSUNEMPBAS', #Unemployment base--persons in civilian labor force (unemployment rate)
    'LINGISO', #Limited English speaking households
    'ACSTOTHH', #Households (for limited English speaking)
    'LESSHS', 
    'ACSEDUCBAS', #Population 25 years and over (use for less than hs)
    'UNDER5', 
    'OVER64',
    'P_LIFEEXPPCT', 
    'ST_ABBREV', 
    'CNTY_NAME'
]

In [11]:
df = pd.read_csv('data/EJSCREEN_2023_BG_with_AS_CNMI_GU_VI.csv', usecols = use_cols, encoding = 'latin1', chunksize = 5000)

cols_agg = {
    'ST_ABBREV': 'first',
    'CNTY_NAME': 'first',
    'P_LIFEEXPPCT': 'mean', #get an average percentile
    'PEOPCOLOR': 'sum',
    'ACSTOTPOP': 'sum',
    'LOWINCOME': 'sum',
    'ACSIPOVBAS': 'sum',
    'UNEMPLOYED': 'sum',
    'ACSUNEMPBAS': 'sum',
    'LINGISO': 'sum', 
    'ACSTOTHH': 'sum',
    'LESSHS': 'sum', 
    'ACSEDUCBAS': 'sum',
    'UNDER5': 'sum', 
    'OVER64': 'sum',
}

output = pd.DataFrame()
for chunk in df:
    
    chunk['County FIPS'] = chunk['ID'].astype(str).str.zfill(12).str[:5] #fill front with 0s in case fips codes should be 12 digits
    chunk_grouped = chunk.groupby('County FIPS').agg(cols_agg) #group within chunk for efficiency, but will need to repeat
    output = pd.concat([chunk_grouped, output])
    #display(output)




In [12]:
#final grouping bc same county could have been in different 'chunks'
final_df = output.groupby(output.index).agg(cols_agg).reset_index()

In [13]:
final_df

,County FIPS,ST_ABBREV,CNTY_NAME,P_LIFEEXPPCT,PEOPCOLOR,ACSTOTPOP,LOWINCOME,ACSIPOVBAS,UNEMPLOYED,ACSUNEMPBAS,LINGISO,ACSTOTHH,LESSHS,ACSEDUCBAS,UNDER5,OVER64
0,00000,MP,Saipan Municipality,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,00780,VI,St. Croix Island,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,01001,AL,Autauga County,72.075000,15668.0,58239.0,17782.0,57790.0,752.0,26623.0,32.0,21856.0,4126.0,39614.0,3318.0,8815.0
3,01003,AL,Baldwin County,56.908257,39583.0,227131.0,57840.0,223772.0,3994.0,108361.0,730.0,87190.0,14555.0,161977.0,12035.0,46805.0
4,01005,AL,Barbour County,84.588235,13991.0,25259.0,11195.0,22250.0,808.0,9369.0,117.0,9088.0,4378.0,17995.0,1320.0,4801.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3218,72145,PR,Vega Baja Municipio,NaN,53453.0,54544.0,40080.0,54242.0,3482.0,19789.0,13350.0,19799.0,9466.0,39632.0,2192.0,11463.0
3219,72147,PR,Vieques Municipio,NaN,7808.0,8317.0,7186.0,8317.0,358.0,2355.0,1782.0,2374.0,1613.0,5970.0,401.0,1904.0
3220,72149,PR,Villalba Municipio,NaN,22289.0,22341.0,17790.0,22207.0,1464.0,7856.0,5780.0,7823.0,3329.0,15523.0,1002.0,4188.0
3221,72151,PR,Yabucoa Municipio,NaN,31020.0,31047.0,24486.0,31042.0,1506.0,9897.0,8645.0,11905.0,6097.0,22690.0,1092.0,6801.0


In [14]:
#renaming and calculating to match eGRID naming

# note that need to divide by different column values as specified in EJScreen technical guide
final_df['Total Population'] = final_df['ACSTOTPOP']
final_df['People of Color (%)'] = (final_df['PEOPCOLOR'] / final_df['ACSTOTPOP']) * 100
final_df['Low Income (%)'] = (final_df['LOWINCOME'] / final_df['ACSIPOVBAS']) * 100 #divide by Population for whom poverty status is determined
final_df['Unemployment Rate (%)'] = (final_df['UNEMPLOYED'] / final_df['ACSUNEMPBAS']) * 100 #Unemployment base--persons in civilian labor force (unemployment rate)
final_df['Limited English Speaking (%)'] = (final_df['LINGISO'] / final_df['ACSTOTHH']) * 100 #Households (for limited English speaking)
final_df['Less Than High School Education (%)'] = (final_df['LESSHS'] / final_df['ACSEDUCBAS']) * 100 #Population 25 years and over (use for less than hs)
final_df['Under Age 5 (%)'] = (final_df['UNDER5'] / final_df['ACSTOTPOP']) * 100
final_df['Over Age 64 (%)'] = (final_df['OVER64'] / final_df['ACSTOTPOP']) * 100
final_df['Plant state abbreviation'] = final_df['ST_ABBREV']
final_df['Plant county name'] = final_df['CNTY_NAME']
final_df['has_plant'] = 0



In [15]:
df_drop = final_df.dropna()
save_df = df_drop[['County FIPS', 'Plant state abbreviation', 'Plant county name', 'has_plant','Total Population', 'People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Limited English Speaking (%)', 'Unemployment Rate (%)']]
save_df

,County FIPS,Plant state abbreviation,Plant county name,has_plant,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%)
2,01001,AL,Autauga County,0,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625
3,01003,AL,Baldwin County,0,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828
4,01005,AL,Barbour County,0,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186
5,01007,AL,Bibb County,0,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819
6,01009,AL,Blount County,0,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723
...,...,...,...,...,...,...,...,...,...,...
3140,56037,WY,Sweetwater County,0,42459.0,21.705645,22.398131,7.294363,1.976946,6.713718
3141,56039,WY,Teton County,0,23319.0,20.073760,21.523236,3.969647,4.889309,2.402089
3142,56041,WY,Uinta County,0,20514.0,12.971629,25.687078,6.430892,1.641694,3.467517
3143,56043,WY,Washakie County,0,7768.0,18.357364,25.391808,5.900793,0.296736,2.519960


In [16]:
save_df.to_csv('data/EJScreen_DEMO23.csv')